# M3 — QASPER RAG + LoRA

Public portfolio edition prepared for GitHub and Databricks. Credentials are read from environment variables; research data and generated artifacts are not committed to Git.


In [ ]:
# !pip installs (safe to re-run)
!pip install -U transformers peft accelerate pandas pyarrow tqdm --quiet

In [ ]:
# Databricks uses Unity Catalog Volumes; no Google Drive mount is required.

In [ ]:
from huggingface_hub import login
login(token=os.environ.get("HF_TOKEN"))

In [ ]:
# ===== User config (edit to your paths if needed) =====
BASE_MODEL   = "mistralai/Mistral-7B-Instruct-v0.3"
LORA_DIR     = "/Volumes/main/default/thesis_project/M3/M3_Test/test_out_M3_Lite_QASPER_1.0"
TEST_JSONL    = "/Volumes/main/default/thesis_project/M3/M3_Test/test_qasper_e5_1.0/qasper_test_topk.jsonl"

# Outputs
OUT_CSV      = "/Volumes/main/default/thesis_project/M3/M3_Test/pred_test_M3_1.4.csv"
OUT_PARQUET  = "/Volumes/main/default/thesis_project/M3/M3_Test/pred_test_M3_1.4.parquet"
OUT_PRED_JSONL = "/Volumes/main/default/thesis_project/M3/M3_Test/pred_test_M3_1.4.jsonl"

# Generation defaults
DO_SAMPLE        = True     # sampling often helps produce fuller sentences (easier to cite)
MAX_NEW_TOKENS   = 224
SEED             = 42
DO_MERGE         = False

In [ ]:
import os, gc, random, numpy as np, torch
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
gc.collect(); torch.cuda.empty_cache()
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

print("Torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

1) Load base model + sanitize & load LoRA

In [ ]:
import os, json, re, shutil
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel, LoraConfig

ALLOWED_KEYS = {
    "r","lora_alpha","lora_dropout","target_modules","fan_in_fan_out","bias",
    "use_dora","use_rslora","init_lora_weights","inference_mode",
    "rank_pattern","alpha_pattern","modules_to_save","layers_to_transform","layers_pattern",
    "task_type","peft_type","auto_mapping","base_model_name_or_path","revision",
    "loftq_config","megablocks"
}

In [ ]:
def sanitize_and_load_lora(base_model_obj, adapter_dir: str) -> PeftModel:
    cfg_path = os.path.join(adapter_dir, "adapter_config.json")
    assert os.path.exists(cfg_path), f"adapter_config.json is missing in: {adapter_dir}"
    with open(cfg_path, "r") as f:
        cfg = json.load(f)
    try: shutil.copy(cfg_path, cfg_path + ".bak_autofix")
    except: pass
    for nested_key in ("lora_config", "dora_config", "corda_config"):
        if nested_key in cfg and isinstance(cfg[nested_key], dict):
            for k, v in cfg.pop(nested_key).items():
                cfg.setdefault(k, v)
            if nested_key in ("dora_config", "corda_config"):
                cfg["use_dora"] = True
    for k in list(cfg.keys()):
        if re.search(r"_config$", k) and k not in ("loftq_config",):
            cfg.pop(k, None)
    cfg["peft_type"] = "LORA"
    cfg.setdefault("task_type", "CAUSAL_LM")
    if not cfg.get("target_modules"):
        cfg["target_modules"] = ["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"]
    clean = {k: v for k, v in cfg.items() if k in ALLOWED_KEYS}
    print("Using cleaned LoRA config keys:", sorted(clean.keys()))
    peft_conf = LoraConfig(**clean)
    model = PeftModel.from_pretrained(base_model_obj, adapter_dir, config=peft_conf)
    return model.eval()

In [ ]:
# Tokenizer（推理：左侧padding更稳）
tok = AutoTokenizer.from_pretrained(LORA_DIR if os.path.isdir(LORA_DIR) else BASE_MODEL, use_fast=True)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
tok.padding_side = "left"
tok.truncation_side = "left"

In [ ]:
# Base model
dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32
base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, torch_dtype=dtype, device_map=None, low_cpu_mem_usage=True
).to("cuda" if torch.cuda.is_available() else "cpu").eval()


In [ ]:
# Inject LoRA（推理只读；可选合并）
model = sanitize_and_load_lora(base, LORA_DIR).to(base.device).eval()
if DO_MERGE:
    try:
        model = model.merge_and_unload()
        print("✓ merged LoRA into base for faster inference")
    except Exception as e:
        print("merge failed (skipping):", e)
model.config.use_cache = True

In [ ]:
# Inspect active PEFT config (safe across PEFT versions)
def inspect_peft(model):
    cfgs = getattr(model, "peft_config", None)
    if cfgs is None:
        print("⚠️ No peft_config found on model"); return
    if isinstance(cfgs, dict):
        name = getattr(model, "active_adapter", None) or next(iter(cfgs.keys()))
        pc = cfgs[name]; print("active_adapter:", name)
    else:
        pc = cfgs; print("active_adapter: <single>")
    print("peft base:", getattr(pc, "base_model_name_or_path", None))
    print("r:", getattr(pc, "r", None),
          "alpha:", getattr(pc, "lora_alpha", None),
          "dropout:", getattr(pc, "lora_dropout", None))
inspect_peft(model)

2) Generation helpers (chat template, decode new-only)

In [ ]:
import re, torch

@torch.no_grad()
def decode_new_tokens(model, tok, prompt: str, *, max_new_tokens: int, do_sample: bool):
    inputs = tok(prompt, return_tensors="pt").to(model.device)
    in_len = inputs["input_ids"].shape[1]
    gen_kwargs = dict(
        max_new_tokens=max_new_tokens,
        do_sample=do_sample,
        repetition_penalty=1.05,
        no_repeat_ngram_size=6,
        eos_token_id=tok.eos_token_id,
        pad_token_id=tok.pad_token_id,
        num_beams=1,
    )
    if do_sample:
        gen_kwargs.update(dict(temperature=0.4, top_p=0.9))
    out = model.generate(**inputs, **gen_kwargs)
    new_tokens = out[0, in_len:]
    text = tok.decode(new_tokens, skip_special_tokens=True).strip()
    return text


In [ ]:
def build_ctx(passages):
    def _c(t): return (t or "").replace("\n"," ").strip()
    return "\n".join(f"[s{i+1}] {_c(p.get('text',''))}" for i, p in enumerate(passages))

3) Answer generator (avoid yes/no; optional rewrite; clean BIBREF)

In [ ]:
@torch.no_grad()
def generate_answer_only(model, tok, passages, question,
                         max_new_tokens=MAX_NEW_TOKENS, do_sample=DO_SAMPLE):
    ctx = build_ctx(passages)
    SYS = (
        "You are a careful research assistant. Use ONLY the provided sources.\n"
        "Write ONE or TWO concise sentences that explicitly state the factual answer "
        "(avoid yes/no or bare lists). If unknown, say \"I don't know.\" "
        "Do not repeat the system or the context."
    )
    msgs = [
        {"role":"system","content":SYS},
        {"role":"user","content":f"[CONTEXT]\n{ctx}\n[/CONTEXT]\n\nQuestion: {question}"},
    ]
    prompt = tok.apply_chat_template(msgs, add_generation_prompt=True, tokenize=False)
    ans = decode_new_tokens(model, tok, prompt, max_new_tokens=max_new_tokens, do_sample=do_sample)
    # 清理
    ans = re.sub(r"<<SYS>>.*?<</SYS>>", "", ans, flags=re.S)
    ans = re.sub(r"\[CONTEXT\].*?\[/CONTEXT\]", "", ans, flags=re.S)
    ans = re.sub(r"\bBIBREF\d+\b", "", ans).strip()
    # 120-word cap
    words = ans.split()
    if len(words) > 120:
        ans = " ".join(words[:120])
    return ans

4) Rerank within sample’s Top-K (lexical F1 + e5-small semantic)

In [ ]:
# Lightweight semantic scorer with e5-small (downloaded once)
USE_E5_SIM = True
E5_MODEL_ID = "intfloat/e5-small-v2"
_e5_tok = _e5_enc = None

In [ ]:
def ensure_e5():
    global _e5_tok, _e5_enc
    if _e5_tok is None or _e5_enc is None:
        from transformers import AutoTokenizer, AutoModel
        dev = model.device
        _e5_tok = AutoTokenizer.from_pretrained(E5_MODEL_ID)
        _e5_enc = AutoModel.from_pretrained(E5_MODEL_ID).to(dev).eval()

def norm_words(s):
    return re.findall(r"[a-zA-Z0-9]+", (s or "").lower())

def overlap_f1(a, b):
    aw, bw = norm_words(a), norm_words(b)
    if not aw or not bw: return 0.0
    aset, bset = set(aw), set(bw)
    inter = len(aset & bset)
    prec = inter / max(1, len(aset)); rec  = inter / max(1, len(bset))
    return 0.0 if prec+rec == 0 else 2*prec*rec/(prec+rec)

In [ ]:
@torch.no_grad()
def e5_cos(a, b):
    ensure_e5()
    t = _e5_tok([f"query: {a}", f"passage: {b}"], padding=True, truncation=True,
                max_length=256, return_tensors="pt").to(_e5_enc.device)
    h = _e5_enc(**t).last_hidden_state[:,0]
    h = torch.nn.functional.normalize(h, p=2, dim=1)
    return float((h[0] @ h[1]).item())

def fused_score(q, p, w_sem=0.6):
    f1 = overlap_f1(q, p)
    if USE_E5_SIM:
        sem = e5_cos(q, p)
        sem01 = (sem + 1) / 2
        return (1 - w_sem) * f1 + w_sem * sem01
    return f1

def rerank_within_topk(passages, question, topm=4, w_sem=0.6):
    scored = []
    for i, p in enumerate(passages, 1):
        txt = (p.get("text","") or "")
        scored.append((fused_score(question, txt, w_sem=w_sem), i, p))
    scored.sort(key=lambda x: x[0], reverse=True)
    return [p for _, _, p in scored[:topm]]

5) Post-attach citations (sentence-wise; fused matching; global fallback)

In [ ]:
def _get_passages_field(s):
    return s.get("topk_passages") or s.get("retrieved") or []

In [ ]:
def sent_split(text: str):
    parts = re.split(r'(?<=[\.!?])\s+|\n+', (text or "").strip())
    return [p.strip() for p in parts if p.strip()]

def attach_citations_sentwise(answer, passages, *,
                              per_sent_top=1, min_score_sent=0.06,
                              global_fallback=True, max_total_cites=2, w_sem=0.6):
    sents = sent_split(answer)
    if not sents:
        return answer
    ptexts = [p.get("text","") or "" for p in passages]

    used = []
    cited_any = False
    out_sents = []

    for s in sents:
        scores = [(fused_score(s, pt, w_sem=w_sem), i+1) for i, pt in enumerate(ptexts)]
        scores.sort(reverse=True)
        picks = [i for sc, i in scores[:per_sent_top] if sc >= min_score_sent]
        if picks:
            cited_any = True
            used.extend(picks)
            out_sents.append(s + " " + " ".join(f"[s{j}]" for j in picks))
        else:
            out_sents.append(s)

    # Global fallback: add one cite to the last sentence if none were attached
    if (not cited_any) and global_fallback and ptexts:
        g_scores = [(fused_score(answer, pt, w_sem=w_sem), i+1) for i, pt in enumerate(ptexts)]
        g_scores.sort(reverse=True)
        if g_scores and g_scores[0][0] >= max(0.05, min_score_sent * 0.7):
            j = g_scores[0][1]
            out_sents[-1] = out_sents[-1] + f" [s{j}]"
            used.append(j)

    # Limit the total distinct cited sources (optional)
    if max_total_cites:
        uniq = []
        for j in used:
            if j not in uniq:
                uniq.append(j)
        if len(uniq) > max_total_cites:
            keep = set(uniq[:max_total_cites])
            def filter_tail(sentence):
                return re.sub(r"\[s(\d+)\]", lambda m: m.group(0) if int(m.group(1)) in keep else "", sentence)
            out_sents = [filter_tail(s) for s in out_sents]

    return " ".join(out_sents)

6) Quick preview (N samples)

In [ ]:
import json, random, textwrap, re

def preview_rerank_then_answer_test(test_jsonl_path, n=5, width=110, topm=4):
    items = [json.loads(l) for l in open(test_jsonl_path,"r",encoding="utf-8")]
    items = [s for s in items if s.get("question") and _get_passages_field(s)]
    random.seed(SEED)
    picks = random.sample(items, min(n, len(items)))
    for idx, s in enumerate(picks, 1):
        q = s["question"]
        passages = rerank_within_topk(_get_passages_field(s), q, topm=topm, w_sem=0.6) or _get_passages_field(s)[:topm]
        ans = generate_answer_only(model, tok, passages, q, max_new_tokens=MAX_NEW_TOKENS, do_sample=DO_SAMPLE)
        cites = []
        for i in range(len(passages)):
            cites.append(f"[s{i+1}]")
        print("="*width)
        print(f"[{idx}/{len(picks)}] Q:", q)
        print("Ans preview:", ans[:min(300,len(ans))], "...")
        print("="*width, "\n")

In [ ]:
preview_rerank_then_answer_test(TEST_JSONL, n=3, topm=4)

7) Full validation inference & save

In [ ]:
import json, pandas as pd
from tqdm.auto import tqdm
import re

rows, out_jsonl = [], open(OUT_PRED_JSONL, "w", encoding="utf-8")
with open(TEST_JSONL, "r", encoding="utf-8") as f:
    for line in tqdm(f, desc="Infer test (rerank → answer)"):
        s = json.loads(line)
        q = s.get("question","").strip()
        topk = _get_passages_field(s)
        if not q or not topk:
            continue

        # 1) Rerank inside sample’s own Top-K
        passages = rerank_within_topk(topk, q, topm=4, w_sem=0.6) or topk[:4]

        # 2) Generate concise factual sentence(s)
        ans = generate_answer_only(model, tok, passages, q, max_new_tokens=MAX_NEW_TOKENS, do_sample=DO_SAMPLE)

        # 3) 简单附上引用（若需要句级引用可沿用你val那套 attach_citations_sentwise；这里先保留原答案）
        ans_cited = ans  # 或者用 attach_citations_sentwise(ans, passages, ...)

        qid = s.get("question_id") or s.get("qid") or ""
        ctxs = [p.get("text","") for p in passages]

        rows.append({
            "question_id": qid,
            "question": q,
            "contexts": ctxs,
            "answer_M3": ans_cited
        })
        out_jsonl.write(json.dumps({"question_id": qid, "answer": ans_cited}, ensure_ascii=False) + "\n")
out_jsonl.close()


df = pd.DataFrame(rows)
df.to_csv(OUT_CSV, index=False)
df.to_parquet(OUT_PARQUET, index=False)
print("✓ Saved predictions:")
print(" •", OUT_CSV)
print(" •", OUT_PARQUET)
print(" •", OUT_PRED_JSONL)
df.head(3)[["question_id","question","answer_M3"]]